In [152]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from CosinorPy import file_parser, cosinor, cosinor1

In [153]:
# Set plotting parameters
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'svg.fonttype': 'none'
})
sns.set(style='whitegrid')

## Clean node files for Cosinor

In [154]:
def subset_asv_table(md_subset, asv_table, sample_col="sample_name", otu_col="#OTU ID"):
    """
    Subset an ASV table based on a metadata subset and remove rows with all NAs.

    Parameters:
    - md_subset: pd.DataFrame, metadata subset
    - asv_table: pd.DataFrame, ASV table with samples as columns
    - sample_col: str, column in metadata containing sample names
    - otu_col: str, column in ASV table with OTU IDs

    Returns:
    - pd.DataFrame with otu_col + existing samples, with rows with all NAs removed
    """
    # Get sample names from metadata
    samples = md_subset[sample_col].tolist()
    
    # Keep only samples that exist in ASV table
    existing_samples = [s for s in samples if s in asv_table.columns]
    
    # Subset ASV table
    subset_table = asv_table[[otu_col] + existing_samples].copy()
    
    # Replace common non-NaN placeholders with actual np.nan
    subset_table[existing_samples] = subset_table[existing_samples].replace(["NA", "", None], np.nan)
    
    # Drop rows where all sample columns are NaN
    subset_table = subset_table.dropna(axis=0, how='all', subset=existing_samples)

    # Drop flat rows (sd == 0) — cosinor cannot fit these and throws RuntimeWarnings
    flat_mask = subset_table[existing_samples].std(axis=1) == 0
    subset_table = subset_table[~flat_mask]
    
    return subset_table


In [155]:
metadata = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/metadata_2026.tsv', sep='\t',low_memory=False)
node = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/feature-table.tsv',sep='\t',low_memory = False)

In [156]:
mdAA = metadata[metadata["title2"] == "Apoe_adlib_11.5"].copy()
mdAT = metadata[metadata["title2"] == "Apoe_TRF_11.5"].copy()
mdLA = metadata[metadata["title2"] == "Ldlr_adlib_11.5"].copy()
mdLT = metadata[metadata["title2"] == "Ldlr_TRF_11.5"].copy()
mdAAl = metadata[metadata["title2"] == "Apoe_adlib_29.5"].copy()
mdATl = metadata[metadata["title2"] == "Apoe_TRF_29.5"].copy()
mdLAl = metadata[metadata["title2"] == "Ldlr_adlib_29.5"].copy()
mdLTl = metadata[metadata["title2"] == "Ldlr_TRF_29.5"].copy()

In [157]:
mtbAA = subset_asv_table(mdAA, node).set_index("#OTU ID")
mtbAT = subset_asv_table(mdAT, node).set_index("#OTU ID")
mtbLA = subset_asv_table(mdLA, node).set_index("#OTU ID")
mtbLT = subset_asv_table(mdLT, node).set_index("#OTU ID")
mtbAAl = subset_asv_table(mdAAl, node).set_index("#OTU ID")
mtbATl = subset_asv_table(mdATl, node).set_index("#OTU ID")
mtbLAl = subset_asv_table(mdLAl, node).set_index("#OTU ID")
mtbLTl = subset_asv_table(mdLTl, node).set_index("#OTU ID")

In [158]:
mtbAA = mtbAA.T.rename_axis("sample_name")
mtbAT = mtbAT.T.rename_axis("sample_name")
mtbLA = mtbLA.T.rename_axis("sample_name")
mtbLT = mtbLT.T.rename_axis("sample_name")
mtbAAl = mtbAAl.T.rename_axis("sample_name")
mtbATl = mtbATl.T.rename_axis("sample_name")
mtbLAl = mtbLAl.T.rename_axis("sample_name")
mtbLTl = mtbLTl.T.rename_axis("sample_name")

In [159]:
mtbAA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/AA_feature-table.csv", index=True)
mtbAT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/AT_feature-table.csv", index=True)
mtbLA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LA_feature-table.csv", index=True)
mtbLT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LT_feature-table.csv", index=True)
mtbAAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/AAl_feature-table.csv", index=True)
mtbATl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/ATl_feature-table.csv", index=True)
mtbLAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LAl_feature-table.csv", index=True)
mtbLTl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LTl_feature-table.csv", index=True)

## Run Cosinor

In [160]:
def fcosinor(joint_file):
    joint_file.zt = joint_file.zt.astype(float)
    joint_file = joint_file.astype(float)
    melted = pd.melt(joint_file, id_vars = 'zt')
    melted.rename(columns={'zt': 'x','value': 'y','variable': 'test'}, inplace = True)
    melted.x = melted.x.astype(float)
    melted.y = melted.y.astype(float)
    cosinor_results = cosinor1.fit_group(melted, period=[24], plot_on = False).sort_values(by=['q'])
    sig = cosinor_results[cosinor_results['q'] < 0.05 ].count()
    print(sig.max())
    return cosinor_results

In [161]:
mtbAA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/AA_feature-table.csv", index_col = 'sample_name',low_memory=False)
mtbAT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/AT_feature-table.csv", index_col = 'sample_name',low_memory=False)
mtbLA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LA_feature-table.csv", index_col = 'sample_name',low_memory=False)
mtbLT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LT_feature-table.csv", index_col = 'sample_name',low_memory=False)
mtbAAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/AAl_feature-table.csv", index_col = 'sample_name',low_memory=False)
mtbATl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/ATl_feature-table.csv", index_col = 'sample_name',low_memory=False)
mtbLAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LAl_feature-table.csv", index_col = 'sample_name',low_memory=False)
mtbLTl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/modern_beta/phyloRPCA-table/LTl_feature-table.csv", index_col = 'sample_name',low_memory=False)

In [162]:
metadata['zt'] = metadata['ZT_time'].str.extract(r'(\d+)').astype('Int64')
metadata = metadata.set_index("sample_name")

In [163]:
#run cosinor
AAr = fcosinor(mtbAA.join(metadata.zt))
ATr = fcosinor(mtbAT.join(metadata.zt))
LAr = fcosinor(mtbLA.join(metadata.zt))
LTr = fcosinor(mtbLT.join(metadata.zt))
AAlr = fcosinor(mtbAAl.join(metadata.zt))
ATlr = fcosinor(mtbATl.join(metadata.zt))
LAlr = fcosinor(mtbLAl.join(metadata.zt))
LTlr = fcosinor(mtbLTl.join(metadata.zt))

0
0


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:985: RuntimeWarning: invalid value encountered in scalar multiply
  b_r = (1 / (1 + (beta_s**2 / beta_r**2))) * (-beta_s / beta_r**2)
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:986: RuntimeWarning: invalid value encountered in scalar multiply
  b_s = (1 / (1 + (beta_s**2 / beta_r**2))) * (1 / beta_r)


0


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:985: RuntimeWarning: invalid value encountered in scalar multiply
  b_r = (1 / (1 + (beta_s**2 / beta_r**2))) * (-beta_s / beta_r**2)
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:986: RuntimeWarning: invalid value encountered in scalar multiply
  b_s = (1 / (1 + (beta_s**2 / beta_r**2))) * (1 / beta_r)


0
0
0
0
0


In [164]:
AAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AA_notnorm.csv", index=True)
ATr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AT_notnorm.csv", index=True)
LAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LA_notnorm.csv", index=True)
LTr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LT_notnorm.csv", index=True)
AAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AAl_notnorm.csv", index=True)
ATlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_ATl_notnorm.csv", index=True)
LAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LAl_notnorm.csv", index=True)
LTlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LTl_notnorm.csv", index=True)

## Clean node RCLR files for Cosinor

In [70]:
def subset_asv_table(md_subset, asv_table, sample_col="sample_name", otu_col="#OTU ID"):
    """
    Subset an ASV table based on a metadata subset and remove rows with all NAs.

    Parameters:
    - md_subset: pd.DataFrame, metadata subset
    - asv_table: pd.DataFrame, ASV table with samples as columns
    - sample_col: str, column in metadata containing sample names
    - otu_col: str, column in ASV table with OTU IDs

    Returns:
    - pd.DataFrame with otu_col + existing samples, with rows with all NAs removed
    """
    # Get sample names from metadata
    samples = md_subset[sample_col].tolist()
    
    # Keep only samples that exist in ASV table
    existing_samples = [s for s in samples if s in asv_table.columns]
    
    # Subset ASV table
    subset_table = asv_table[[otu_col] + existing_samples].copy()
    
    # Replace common non-NaN placeholders with actual np.nan
    subset_table[existing_samples] = subset_table[existing_samples].replace(["NA", "", None], np.nan)
    
    # Drop rows where all sample columns are NaN
    subset_table = subset_table.dropna(axis=0, how='all', subset=existing_samples)
    
    return subset_table

In [6]:
metadata = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/metadata_2026.tsv', sep='\t',low_memory=False)
asv_rclr = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/feature-table.tsv',sep='\t',low_memory = False)

In [7]:
mdAA = metadata[metadata["title2"] == "Apoe_adlib_11.5"].copy()
mdAT = metadata[metadata["title2"] == "Apoe_TRF_11.5"].copy()
mdLA = metadata[metadata["title2"] == "Ldlr_adlib_11.5"].copy()
mdLT = metadata[metadata["title2"] == "Ldlr_TRF_11.5"].copy()
mdAAl = metadata[metadata["title2"] == "Apoe_adlib_29.5"].copy()
mdATl = metadata[metadata["title2"] == "Apoe_TRF_29.5"].copy()
mdLAl = metadata[metadata["title2"] == "Ldlr_adlib_29.5"].copy()
mdLTl = metadata[metadata["title2"] == "Ldlr_TRF_29.5"].copy()

In [8]:
mtbAA = subset_asv_table(mdAA, asv_rclr).set_index("#OTU ID")
mtbAT = subset_asv_table(mdAT, asv_rclr).set_index("#OTU ID")
mtbLA = subset_asv_table(mdLA, asv_rclr).set_index("#OTU ID")
mtbLT = subset_asv_table(mdLT, asv_rclr).set_index("#OTU ID")
mtbAAl = subset_asv_table(mdAAl, asv_rclr).set_index("#OTU ID")
mtbATl = subset_asv_table(mdATl, asv_rclr).set_index("#OTU ID")
mtbLAl = subset_asv_table(mdLAl, asv_rclr).set_index("#OTU ID")
mtbLTl = subset_asv_table(mdLTl, asv_rclr).set_index("#OTU ID")

In [9]:
mtbAA = mtbAA.T.rename_axis("sample_name")
mtbAT = mtbAT.T.rename_axis("sample_name")
mtbLA = mtbLA.T.rename_axis("sample_name")
mtbLT = mtbLT.T.rename_axis("sample_name")
mtbAAl = mtbAAl.T.rename_axis("sample_name")
mtbATl = mtbATl.T.rename_axis("sample_name")
mtbLAl = mtbLAl.T.rename_axis("sample_name")
mtbLTl = mtbLTl.T.rename_axis("sample_name")

In [102]:
mtbAA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/AA_feature-table_rclr.csv", index=True)
mtbAT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/AT_feature-table_rclr.csv", index=True)
mtbLA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LA_feature-table_rclr.csv", index=True)
mtbLT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LT_feature-table_rclr.csv", index=True)
mtbAAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/AAl_feature-table_rclr.csv", index=True)
mtbATl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/ATl_feature-table_rclr.csv", index=True)
mtbLAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LAl_feature-table_rclr.csv", index=True)
mtbLTl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LTl_feature-table_rclr.csv", index=True)

## Run Cosinor

In [10]:
def fcosinor(joint_file):
    joint_file.zt = joint_file.zt.astype(float)
    joint_file = joint_file.astype(float)
    melted = pd.melt(joint_file, id_vars = 'zt')
    melted.rename(columns={'zt': 'x','value': 'y','variable': 'test'}, inplace = True)
    melted.x = melted.x.astype(float)
    melted.y = melted.y.astype(float)
    cosinor_results = cosinor1.fit_group(melted, period=[24], plot_on = False).sort_values(by=['q'])
    sig = cosinor_results[cosinor_results['q'] < 0.05 ].count()
    print(sig.max())
    return cosinor_results

In [11]:
mtbAA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/AA_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbAT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/AT_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LA_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LT_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbAAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/AAl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbATl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/ATl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LAl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLTl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr_phylo/rclr_table/LTl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)

In [12]:
metadata['zt'] = metadata['ZT_time'].str.extract(r'(\d+)').astype('Int64')
metadata = metadata.set_index("sample_name")

In [13]:
#run cosinor
AAr = fcosinor(mtbAA.join(metadata.zt))
ATr = fcosinor(mtbAT.join(metadata.zt))
LAr = fcosinor(mtbLA.join(metadata.zt))
LTr = fcosinor(mtbLT.join(metadata.zt))
AAlr = fcosinor(mtbAAl.join(metadata.zt))
ATlr = fcosinor(mtbATl.join(metadata.zt))
LAlr = fcosinor(mtbLAl.join(metadata.zt))
LTlr = fcosinor(mtbLTl.join(metadata.zt))

/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packag

20


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10

9


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplico

...

/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: invalid value encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/regression/linear_model.py:1717: RuntimeWarning: invalid value encountered in scalar divide
  return np.dot(wresid, wresid) / self.df_resid
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/env

43


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))


97


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:356: RuntimeWarning: invalid value encountered in accumulate
  pvals_corrected = np.minimum.accumulate(pvals_corrected_raw[::-1])[::-1]
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:358: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:268: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1


In [107]:
AAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AA.csv", index=True)
ATr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AT.csv", index=True)
LAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LA.csv", index=True)
LTr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LT.csv", index=True)
AAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AAl.csv", index=True)
ATlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_ATl.csv", index=True)
LAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LAl.csv", index=True)
LTlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LTl.csv", index=True)

## Clean ASV RCLR files for Cosinor

In [106]:
metadata = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/metadata_2026.tsv', sep='\t',low_memory=False)
asv_rclr = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/feature-table.tsv',sep='\t',low_memory = False)

In [107]:
mdAA = metadata[metadata["title2"] == "Apoe_adlib_11.5"].copy()
mdAT = metadata[metadata["title2"] == "Apoe_TRF_11.5"].copy()
mdLA = metadata[metadata["title2"] == "Ldlr_adlib_11.5"].copy()
mdLT = metadata[metadata["title2"] == "Ldlr_TRF_11.5"].copy()
mdAAl = metadata[metadata["title2"] == "Apoe_adlib_29.5"].copy()
mdATl = metadata[metadata["title2"] == "Apoe_TRF_29.5"].copy()
mdLAl = metadata[metadata["title2"] == "Ldlr_adlib_29.5"].copy()
mdLTl = metadata[metadata["title2"] == "Ldlr_TRF_29.5"].copy()

In [108]:
mtbAA = subset_asv_table(mdAA, asv_rclr).set_index("#OTU ID")
mtbAT = subset_asv_table(mdAT, asv_rclr).set_index("#OTU ID")
mtbLA = subset_asv_table(mdLA, asv_rclr).set_index("#OTU ID")
mtbLT = subset_asv_table(mdLT, asv_rclr).set_index("#OTU ID")
mtbAAl = subset_asv_table(mdAAl, asv_rclr).set_index("#OTU ID")
mtbATl = subset_asv_table(mdATl, asv_rclr).set_index("#OTU ID")
mtbLAl = subset_asv_table(mdLAl, asv_rclr).set_index("#OTU ID")
mtbLTl = subset_asv_table(mdLTl, asv_rclr).set_index("#OTU ID")

In [109]:
mtbAA = mtbAA.T.rename_axis("sample_name")
mtbAT = mtbAT.T.rename_axis("sample_name")
mtbLA = mtbLA.T.rename_axis("sample_name")
mtbLT = mtbLT.T.rename_axis("sample_name")
mtbAAl = mtbAAl.T.rename_axis("sample_name")
mtbATl = mtbATl.T.rename_axis("sample_name")
mtbLAl = mtbLAl.T.rename_axis("sample_name")
mtbLTl = mtbLTl.T.rename_axis("sample_name")

In [102]:
mtbAA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/AA_feature-table_rclr.csv", index=True)
mtbAT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/AT_feature-table_rclr.csv", index=True)
mtbLA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LA_feature-table_rclr.csv", index=True)
mtbLT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LT_feature-table_rclr.csv", index=True)
mtbAAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/AAl_feature-table_rclr.csv", index=True)
mtbATl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/ATl_feature-table_rclr.csv", index=True)
mtbLAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LAl_feature-table_rclr.csv", index=True)
mtbLTl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LTl_feature-table_rclr.csv", index=True)

## Run Cosinor

In [110]:
def fcosinor(joint_file):
    joint_file.zt = joint_file.zt.astype(float)
    joint_file = joint_file.astype(float)
    melted = pd.melt(joint_file, id_vars = 'zt')
    melted.rename(columns={'zt': 'x','value': 'y','variable': 'test'}, inplace = True)
    melted.x = melted.x.astype(float)
    melted.y = melted.y.astype(float)
    cosinor_results = cosinor1.fit_group(melted, period=[24], plot_on = False).sort_values(by=['q'])
    sig = cosinor_results[cosinor_results['p'] < 0.05 ].count()
    print(sig.max())
    return cosinor_results

In [111]:
mtbAA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/AA_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbAT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/AT_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LA_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LT_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbAAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/AAl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbATl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/ATl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LAl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)
mtbLTl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_rclr/rclr_table/LTl_feature-table_rclr.csv", index_col = 'sample_name',low_memory=False)

In [112]:
metadata['zt'] = metadata['ZT_time'].str.extract(r'(\d+)').astype('Int64')
metadata = metadata.set_index("sample_name")

In [113]:
#run cosinor
AAr = fcosinor(mtbAA.join(metadata.zt))
ATr = fcosinor(mtbAT.join(metadata.zt))
LAr = fcosinor(mtbLA.join(metadata.zt))
LTr = fcosinor(mtbLT.join(metadata.zt))
AAlr = fcosinor(mtbAAl.join(metadata.zt))
ATlr = fcosinor(mtbATl.join(metadata.zt))
LAlr = fcosinor(mtbLAl.join(metadata.zt))
LTlr = fcosinor(mtbLTl.join(metadata.zt))

/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:356: RuntimeWarning: invalid value encountered in accumulate
  pvals_corrected = np.minimum.accumulate(pvals_corrected_raw[::-1])[::-1]
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime

8


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/base/model.py:1527: RuntimeWarning: invalid value encountered in multiply
  cov_p = self.normalized_cov_params * scale
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/pyth

13


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:356: RuntimeWarning: invalid value encountered in accumulate
  pvals_corrected = np.minimum.accumulate(pvals_corrected_raw[::-1])[::-1]
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:358: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:268: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1


...

/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:356: RuntimeWarning: invalid value encountered in accumulate
  pvals_corrected = np.minimum.accumulate(pvals_corrected_raw[::-1])[::-1]
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:358: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:268: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1


16


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:996: RuntimeWarning: invalid value encountered in sqrt
  se_trans_only =np.sqrt(np.diag(cov_trans))


20


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:356: RuntimeWarning: invalid value encountered in accumulate
  pvals_corrected = np.minimum.accumulate(pvals_corrected_raw[::-1])[::-1]
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:358: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/statsmodels/stats/multitest.py:268: RuntimeWarning: invalid value encountered in greater
  pvals_corrected[pvals_corrected>1] = 1


In [107]:
AAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_AA.csv", index=True)
ATr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_AT.csv", index=True)
LAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LA.csv", index=True)
LTr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LT.csv", index=True)
AAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_AAl.csv", index=True)
ATlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_ATl.csv", index=True)
LAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LAl.csv", index=True)
LTlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LTl.csv", index=True)

## Clean ASV CLR files for Cosinor

In [165]:
metadata = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/metadata_2026.tsv', sep='\t',low_memory=False)
asv_clr = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/feature-table.tsv',sep='\t',low_memory = False)

In [166]:
mdAA = metadata[metadata["title2"] == "Apoe_adlib_11.5"].copy()
mdAT = metadata[metadata["title2"] == "Apoe_TRF_11.5"].copy()
mdLA = metadata[metadata["title2"] == "Ldlr_adlib_11.5"].copy()
mdLT = metadata[metadata["title2"] == "Ldlr_TRF_11.5"].copy()
mdAAl = metadata[metadata["title2"] == "Apoe_adlib_29.5"].copy()
mdATl = metadata[metadata["title2"] == "Apoe_TRF_29.5"].copy()
mdLAl = metadata[metadata["title2"] == "Ldlr_adlib_29.5"].copy()
mdLTl = metadata[metadata["title2"] == "Ldlr_TRF_29.5"].copy()

In [167]:
mtbAA = subset_asv_table(mdAA, asv_clr).set_index("#OTU ID")
mtbAT = subset_asv_table(mdAT, asv_clr).set_index("#OTU ID")
mtbLA = subset_asv_table(mdLA, asv_clr).set_index("#OTU ID")
mtbLT = subset_asv_table(mdLT, asv_clr).set_index("#OTU ID")
mtbAAl = subset_asv_table(mdAAl, asv_clr).set_index("#OTU ID")
mtbATl = subset_asv_table(mdATl, asv_clr).set_index("#OTU ID")
mtbLAl = subset_asv_table(mdLAl, asv_clr).set_index("#OTU ID")
mtbLTl = subset_asv_table(mdLTl, asv_clr).set_index("#OTU ID")

In [168]:
mtbAA = mtbAA.T.rename_axis("sample_name")
mtbAT = mtbAT.T.rename_axis("sample_name")
mtbLA = mtbLA.T.rename_axis("sample_name")
mtbLT = mtbLT.T.rename_axis("sample_name")
mtbAAl = mtbAAl.T.rename_axis("sample_name")
mtbATl = mtbATl.T.rename_axis("sample_name")
mtbLAl = mtbLAl.T.rename_axis("sample_name")
mtbLTl = mtbLTl.T.rename_axis("sample_name")

In [169]:
mtbAA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/AA_feature-table_clr.csv", index=True)
mtbAT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/AT_feature-table_clr.csv", index=True)
mtbLA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LA_feature-table_clr.csv", index=True)
mtbLT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LT_feature-table_clr.csv", index=True)
mtbAAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/AAl_feature-table_clr.csv", index=True)
mtbATl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/ATl_feature-table_clr.csv", index=True)
mtbLAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LAl_feature-table_clr.csv", index=True)
mtbLTl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LTl_feature-table_clr.csv", index=True)

## Run Cosinor

In [170]:
def fcosinor(joint_file):
    joint_file.zt = joint_file.zt.astype(float)
    joint_file = joint_file.astype(float)
    melted = pd.melt(joint_file, id_vars = 'zt')
    melted.rename(columns={'zt': 'x','value': 'y','variable': 'test'}, inplace = True)
    melted.x = melted.x.astype(float)
    melted.y = melted.y.astype(float)
    cosinor_results = cosinor1.fit_group(melted, period=[24], plot_on = False).sort_values(by=['q'])
    sig = cosinor_results[cosinor_results['p'] < 0.05 ].count()
    print(sig.max())
    return cosinor_results

In [171]:
mtbAA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/AA_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)
mtbAT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/AT_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)
mtbLA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LA_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)
mtbLT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LT_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)
mtbAAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/AAl_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)
mtbATl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/ATl_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)
mtbLAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LAl_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)
mtbLTl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_clr/clr_table/LTl_feature-table_clr.csv", index_col = 'sample_name',low_memory=False)

In [172]:
metadata['zt'] = metadata['ZT_time'].str.extract(r'(\d+)').astype('Int64')
metadata = metadata.set_index("sample_name")

In [173]:
#run cosinor
AAr = fcosinor(mtbAA.join(metadata.zt))
ATr = fcosinor(mtbAT.join(metadata.zt))
LAr = fcosinor(mtbLA.join(metadata.zt))
LTr = fcosinor(mtbLT.join(metadata.zt))
AAlr = fcosinor(mtbAAl.join(metadata.zt))
ATlr = fcosinor(mtbATl.join(metadata.zt))
LAlr = fcosinor(mtbLAl.join(metadata.zt))
LTlr = fcosinor(mtbLTl.join(metadata.zt))

8
14
21
12
1752
10
8
28


In [105]:
AAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_AA_clr.csv", index=True)
ATr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_AT_clr.csv", index=True)
LAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LA_clr.csv", index=True)
LTr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LT_clr.csv", index=True)
AAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_AAl_clr.csv", index=True)
ATlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_ATl_clr.csv", index=True)
LAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LAl_clr.csv", index=True)
LTlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/asv_analysis/cosinor_LTl_clr.csv", index=True)

## clean norm node files to run consinor

In [142]:
def subset_asv_table(md_subset, asv_table, sample_col="sample_name", otu_col="#OTU ID"):
    """
    Subset an ASV table based on a metadata subset and remove rows with all NAs.
    Parameters:
    - md_subset: pd.DataFrame, metadata subset
    - asv_table: pd.DataFrame, ASV table with samples as columns
    - sample_col: str, column in metadata containing sample names
    - otu_col: str, column in ASV table with OTU IDs
    Returns:
    - pd.DataFrame with otu_col + existing samples, with rows with all NAs removed
      and flat rows (zero variance) removed
    """
    samples = md_subset[sample_col].tolist()
    existing_samples = [s for s in samples if s in asv_table.columns]
    
    subset_table = asv_table[[otu_col] + existing_samples].copy()
    
    # Replace common non-NaN placeholders with actual np.nan
    subset_table[existing_samples] = subset_table[existing_samples].replace(["NA", "", None], np.nan)
    
    # Drop rows where all sample columns are NaN
    subset_table = subset_table.dropna(axis=0, how='all', subset=existing_samples)

    # Drop flat rows (sd == 0) — cosinor cannot fit these and throws RuntimeWarnings
    flat_mask = subset_table[existing_samples].std(axis=1) == 0
    subset_table = subset_table[~flat_mask]

    return subset_table

In [143]:
metadata = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/metadata_2026.tsv', sep='\t',low_memory=False)
noden = pd.read_csv('/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm.tsv',sep='\t',low_memory = False)

In [144]:
mdAA = metadata[metadata["title2"] == "Apoe_adlib_11.5"].copy()
mdAT = metadata[metadata["title2"] == "Apoe_TRF_11.5"].copy()
mdLA = metadata[metadata["title2"] == "Ldlr_adlib_11.5"].copy()
mdLT = metadata[metadata["title2"] == "Ldlr_TRF_11.5"].copy()
mdAAl = metadata[metadata["title2"] == "Apoe_adlib_29.5"].copy()
mdATl = metadata[metadata["title2"] == "Apoe_TRF_29.5"].copy()
mdLAl = metadata[metadata["title2"] == "Ldlr_adlib_29.5"].copy()
mdLTl = metadata[metadata["title2"] == "Ldlr_TRF_29.5"].copy()

In [145]:
mtbAA = subset_asv_table(mdAA, noden).set_index("#OTU ID")
mtbAT = subset_asv_table(mdAT, noden).set_index("#OTU ID")
mtbLA = subset_asv_table(mdLA, noden).set_index("#OTU ID")
mtbLT = subset_asv_table(mdLT, noden).set_index("#OTU ID")
mtbAAl = subset_asv_table(mdAAl, noden).set_index("#OTU ID")
mtbATl = subset_asv_table(mdATl, noden).set_index("#OTU ID")
mtbLAl = subset_asv_table(mdLAl, noden).set_index("#OTU ID")
mtbLTl = subset_asv_table(mdLTl, noden).set_index("#OTU ID")

In [146]:
mtbAA = mtbAA.T.rename_axis("sample_name")
mtbAT = mtbAT.T.rename_axis("sample_name")
mtbLA = mtbLA.T.rename_axis("sample_name")
mtbLT = mtbLT.T.rename_axis("sample_name")
mtbAAl = mtbAAl.T.rename_axis("sample_name")
mtbATl = mtbATl.T.rename_axis("sample_name")
mtbLAl = mtbLAl.T.rename_axis("sample_name")
mtbLTl = mtbLTl.T.rename_axis("sample_name")

In [147]:
mtbAA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/AA_feature-table_nn.csv", index=True)
mtbAT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/AT_feature-table_nn.csv", index=True)
mtbLA.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LA_feature-table_nn.csv", index=True)
mtbLT.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LT_feature-table_nn.csv", index=True)
mtbAAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/AAl_feature-table_nn.csv", index=True)
mtbATl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/ATl_feature-table_nn.csv", index=True)
mtbLAl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LAl_feature-table_nn.csv", index=True)
mtbLTl.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LTl_feature-table_nn.csv", index=True)

## Run Cosinor

In [148]:
def fcosinor(joint_file):
    joint_file.zt = joint_file.zt.astype(float)
    joint_file = joint_file.astype(float)
    melted = pd.melt(joint_file, id_vars = 'zt')
    melted.rename(columns={'zt': 'x','value': 'y','variable': 'test'}, inplace = True)
    melted.x = melted.x.astype(float)
    melted.y = melted.y.astype(float)
    cosinor_results = cosinor1.fit_group(melted, period=[24], plot_on = False).sort_values(by=['q'])
    sig = cosinor_results[cosinor_results['q'] < 0.1 ].count()
    print(sig.max())
    return cosinor_results

In [149]:
mtbAA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/AA_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)
mtbAT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/AT_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)
mtbLA = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LA_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)
mtbLT = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LT_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)
mtbAAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/AAl_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)
mtbATl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/ATl_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)
mtbLAl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LAl_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)
mtbLTl = pd.read_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/gg2_feature-table_filt11400_nodenorm/LTl_feature-table_nn.csv", index_col = 'sample_name',low_memory=False)

In [150]:
metadata['zt'] = metadata['ZT_time'].str.extract(r'(\d+)').astype('Int64')
metadata = metadata.set_index("sample_name")

In [151]:
#run cosinor
AAr = fcosinor(mtbAA.join(metadata.zt))
ATr = fcosinor(mtbAT.join(metadata.zt))
LAr = fcosinor(mtbLA.join(metadata.zt))
LTr = fcosinor(mtbLT.join(metadata.zt))
AAlr = fcosinor(mtbAAl.join(metadata.zt))
ATlr = fcosinor(mtbATl.join(metadata.zt))
LAlr = fcosinor(mtbLAl.join(metadata.zt))
LTlr = fcosinor(mtbLTl.join(metadata.zt))

/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:239: RuntimeWarning: invalid value encountered in scalar divide
  acr[i] = np.pi*(-1)+(-1*np.arctan(np.abs(sss / rrr)))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:983: RuntimeWarning: invalid value encountered in scalar multiply
  a_r = (beta_r**2 + beta_s**2)**(-0.5) * beta_r
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:984: RuntimeWarning: invalid value encountered in scalar multiply
  a_s = (beta_r**2 + beta_s**2)**(-0.5) * beta_s
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:985: RuntimeWarning: invalid value encountered in scalar divide
  b_r = (1 / (1 + (beta_s**2 / beta_r**2))) * (-beta_s / beta_r**2)
/mnt/zarrinpar/Pyn

0


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:239: RuntimeWarning: invalid value encountered in scalar divide
  acr[i] = np.pi*(-1)+(-1*np.arctan(np.abs(sss / rrr)))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:983: RuntimeWarning: invalid value encountered in scalar multiply
  a_r = (beta_r**2 + beta_s**2)**(-0.5) * beta_r
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:984: RuntimeWarning: invalid value encountered in scalar multiply
  a_s = (beta_r**2 + beta_s**2)**(-0.5) * beta_s
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:985: RuntimeWarning: invalid value encountered in scalar divide
  b_r = (1 / (1 + (beta_s**2 / beta_r**2))) * (-beta_s / beta_r**2)
/mnt/zarrinpar/Pyn

0


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:239: RuntimeWarning: invalid value encountered in scalar divide
  acr[i] = np.pi*(-1)+(-1*np.arctan(np.abs(sss / rrr)))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:983: RuntimeWarning: invalid value encountered in scalar multiply
  a_r = (beta_r**2 + beta_s**2)**(-0.5) * beta_r
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:984: RuntimeWarning: invalid value encountered in scalar multiply
  a_s = (beta_r**2 + beta_s**2)**(-0.5) * beta_s
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:985: RuntimeWarning: invalid value encountered in scalar divide
  b_r = (1 / (1 + (beta_s**2 / beta_r**2))) * (-beta_s / beta_r**2)
/mnt/zarrinpar/Pyn

0
0
0


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:239: RuntimeWarning: invalid value encountered in scalar divide
  acr[i] = np.pi*(-1)+(-1*np.arctan(np.abs(sss / rrr)))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:983: RuntimeWarning: invalid value encountered in scalar multiply
  a_r = (beta_r**2 + beta_s**2)**(-0.5) * beta_r
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:984: RuntimeWarning: invalid value encountered in scalar multiply
  a_s = (beta_r**2 + beta_s**2)**(-0.5) * beta_s
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:985: RuntimeWarning: invalid value encountered in scalar divide
  b_r = (1 / (1 + (beta_s**2 / beta_r**2))) * (-beta_s / beta_r**2)
/mnt/zarrinpar/Pyn

0


/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:239: RuntimeWarning: invalid value encountered in scalar divide
  acr[i] = np.pi*(-1)+(-1*np.arctan(np.abs(sss / rrr)))
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:983: RuntimeWarning: invalid value encountered in scalar multiply
  a_r = (beta_r**2 + beta_s**2)**(-0.5) * beta_r
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:984: RuntimeWarning: invalid value encountered in scalar multiply
  a_s = (beta_r**2 + beta_s**2)**(-0.5) * beta_s
/mnt/zarrinpar/Pynchon/opt/anaconda3/envs/cosinor-qiime2-amplicon-2024.10/lib/python3.10/site-packages/CosinorPy/cosinor1.py:985: RuntimeWarning: invalid value encountered in scalar divide
  b_r = (1 / (1 + (beta_s**2 / beta_r**2))) * (-beta_s / beta_r**2)
/mnt/zarrinpar/Pyn

3
72


In [24]:
AAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AA_nn.csv", index=True)
ATr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AT_nn.csv", index=True)
LAr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LA_nn.csv", index=True)
LTr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LT_nn.csv", index=True)
AAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_AAl_nn.csv", index=True)
ATlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_ATl_nn.csv", index=True)
LAlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LAl_nn.csv", index=True)
LTlr.to_csv("/mnt/zarrinpar/scratch/sfloresr/maria_apoe/2026_reprocess_16s/data/cosinor/node_analysis/cosinor_LTl_nn.csv", index=True)